In [ ]:
# loading base model
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "dice-research/lola_v1"
model = AutoModelForCausalLM.from_pretrained(base_model_name,trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(base_model_name,trust_remote_code=True)

In [ ]:
#model

In [ ]:
from peft import PeftModel

device = 'cuda:1'
# Load the LoRA adapter
lora_model_name = "./lola_alpaca_multilingual_peft/"
model = PeftModel.from_pretrained(model, lora_model_name).to(device)

In [ ]:
#model

In [ ]:
PROMPT_DICT = {
    "prompt_input": (
        "Below is an instruction that describes a task, paired with an input that provides further context. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:"
    ),
    "prompt_no_input": (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        "### Instruction:\n{instruction}\n\n### Response:"
    ),
}

In [ ]:
def generate_response(instruction, input_text="", max_length=200, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.15):
    """
    Generate a response based on the Alpaca-style instruction format using sampling-based decoding.
    
    Args:
        instruction (str): The instruction for the model.
        input_text (str): Optional input text related to the instruction.
        max_length (int): The maximum length of the response.
        temperature (float): The temperature for sampling. Lower values make output more deterministic.
        top_k (int): Top-k sampling parameter. Keeps the top-k probable tokens.
        top_p (float): Top-p (nucleus) sampling parameter. Keeps tokens within a cumulative probability p.
    
    Returns:
        str: The generated response from the model.
    """
    # Choose the correct template depending on whether input_text is empty
    if input_text and input_text.strip():
        prompt = PROMPT_DICT["prompt_input"].format(instruction=instruction, input=input_text)
    else:
        prompt = PROMPT_DICT["prompt_no_input"].format(instruction=instruction)

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate response with sampling-based decoding
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # Decode the response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract the response part
    response_start = generated_text.find("### Response:") + len("### Response:")
    response = generated_text[response_start:].strip()
    
    return response

In [ ]:
instruction = "Give tips on staying healthy."
input_text = ""

# simple-1
response = generate_response(
    instruction=instruction,
    input_text=input_text,
    max_length=512,
    temperature=0.7,
    top_k=20,
    top_p=0.9,
    repetition_penalty=1.15
)

print("Generated Response:")
print(response)